# Observability and Debugging Multi-Agent Runs

When a Notes API team returns a wrong answer or runs too long, you need **visibility**: which agent spoke, what tools ran, why the team stopped, and where speaker selection failed. AutoGen 0.4+ AgentChat exposes **`Console` streaming**, rich **`TaskResult`** objects, typed **messages**, and **`CancellationToken`** for cooperative cancel.

This notebook covers Console UI streaming, reading `TaskResult`, message types (`TextMessage`, `ToolCallSummaryMessage`), logging agent traces, debugging speaker selection failures, `CancellationToken`, and an optional LangSmith sketch.

Prerequisites: **08. GroupChat and Multi-Agent Teams**, **09. GroupChatManager and Speaker Selection**, **13. Termination Conditions and Guardrails**.


In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import json
import os
import re
from dataclasses import dataclass, field
from typing import Any, Sequence

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("autogen-agentchat: pip install autogen-agentchat autogen-ext[openai]")

print("asyncio loop ready — use await in notebook cells")


In [ ]:
# Notes API documentation corpus (shared across 03. AutoGen/ track)
NOTES_CORPUS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]

BLOCKED_TERMS = {"password", "secret", "api key", "private key", "credential"}
APPROVE_TOKEN = "APPROVE"
TERMINATE_TOKEN = "TERMINATE"


def keyword_search(query: str, k: int = 2) -> str:
    """Simple keyword retrieval over NOTES_CORPUS."""
    tokens = set(query.lower().split())
    scored = [(len(tokens & set(d["text"].lower().split())), d) for d in NOTES_CORPUS]
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [d for s, d in scored if s > 0][:k] or [NOTES_CORPUS[0]]
    return "\n".join(f"[{d['id']}] {d['text']}" for d in top)


def is_blocked(text: str) -> bool:
    """Pre-flight guard — mirror 02. LangGraph/14 guard node."""
    lower = text.lower()
    return any(term in lower for term in BLOCKED_TERMS)


print("corpus chunks:", len(NOTES_CORPUS))
print("sample search:", keyword_search("JWT bearer")[:80], "...")


In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient


def make_model_client(model: str = "gpt-4o-mini", temperature: float = 0.2) -> OpenAIChatCompletionClient:
    return OpenAIChatCompletionClient(
        model=model,
        api_key=OPENAI_API_KEY,
        temperature=temperature,
    )


model_client = make_model_client()
print("model client:", model_client)


---

## 1. Observability Layers

```
team.run_stream(task)
    │
    ├─► Console UI          live labeled output per agent
    ├─► TaskResult          final messages + stop_reason
    ├─► message types       TextMessage, ToolCall*, events
    ├─► custom logger       wrap stream → JSON traces
    ├─► CancellationToken   cooperative cancel
    └─► LangSmith (optional) cross-run trace UI
```

| Layer | Best for | Cost |
|-------|----------|------|
| **`Console`** | Notebook debugging | Free |
| **`TaskResult`** | API response payload | Free |
| **Custom trace list** | Unit tests, audit log | Free |
| **LangSmith** | Staging/prod replay | SaaS |


---

## 2. Build a Debuggable Notes Team


In [ ]:
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.base import TaskResult

docs_writer = AssistantAgent(
    "docs_writer",
    model_client=model_client,
    system_message="Answer Notes API questions using provided context. Be concise.",
)
critic = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message="Review answers. Say APPROVE when correct.",
)
router = AssistantAgent(
    "router",
    model_client=model_client,
    system_message="You route tasks. Never answer directly.",
)

round_robin = RoundRobinGroupChat(
    [docs_writer, critic],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8),
)
print("teams ready")


---

## 3. Console UI Streaming

`Console` wraps `run_stream()` and prints labeled messages as they arrive:


In [ ]:
from autogen_agentchat.ui import Console

STREAM_DEMO = """
# Live stream with pretty console output:
task = "How do I authenticate with JWT? Context: " + keyword_search("JWT")
result = await Console(round_robin.run_stream(task=task))

# Console prints:
# ---------- user ----------
# How do I authenticate...
# ---------- docs_writer ----------
# Send Authorization: Bearer <token> header [c3]
# ---------- critic ----------
# Correct. APPROVE
# ---------- Summary ----------
# Number of messages: 3
# Finish reason: Text 'APPROVE' mentioned
"""
print(STREAM_DEMO)


Use `Console` in notebooks; in production, write your own stream consumer (next section).


---

## 4. Reading `TaskResult`

`run()` and `run_stream()` return `TaskResult` with `messages` and `stop_reason`:


In [ ]:
def summarize_task_result(result: TaskResult) -> dict:
    """Flatten TaskResult for API responses (used in 16)."""
    speakers = []
    total_prompt = total_completion = 0
    for msg in result.messages:
        speakers.append({"source": msg.source, "type": type(msg).__name__})
        if getattr(msg, "models_usage", None):
            total_prompt += msg.models_usage.prompt_tokens
            total_completion += msg.models_usage.completion_tokens
    last_text = ""
    for msg in reversed(result.messages):
        if isinstance(msg, TextMessage) and msg.source != "user":
            last_text = msg.content
            break
    return {
        "stop_reason": result.stop_reason,
        "message_count": len(result.messages),
        "speakers": speakers,
        "last_agent_text": last_text[:200],
        "tokens": {"prompt": total_prompt, "completion": total_completion},
    }


# Teaching example — simulated result
sample = TaskResult(
    messages=[
        TextMessage(source="user", content="JWT?"),
        TextMessage(source="docs_writer", content="Use Bearer header [c3]"),
        TextMessage(source="critic", content="APPROVE"),
    ],
    stop_reason="Text 'APPROVE' mentioned",
)
print(json.dumps(summarize_task_result(sample), indent=2))


---

## 5. Message Types

| Type | Purpose |
|------|---------|
| `TextMessage` | User text or agent LLM reply |
| `ToolCallRequestEvent` | Agent requested tool calls |
| `ToolCallExecutionEvent` | Tool results |
| `ToolCallSummaryMessage` | Condensed tool output for history |
| `StopMessage` | Emitted by termination conditions |
| `HandoffMessage` | Agent-to-agent handoff (Swarm) |


In [ ]:
from autogen_agentchat.messages import ToolCallSummaryMessage

MESSAGE_TYPE_DEMO = [
    TextMessage(source="docs_writer", content="Searching docs..."),
    ToolCallSummaryMessage(source="docs_writer", content="[c2] Alembic migrations..."),
    TextMessage(source="critic", content="APPROVE"),
]

for m in MESSAGE_TYPE_DEMO:
    print(f"{type(m).__name__:28} source={m.source:12} len={len(m.content)}")


`ToolCallSummaryMessage` is what you often log for RAG/tool teams — it collapses verbose tool I/O into one history line.


---

## 6. Custom Stream Logger — Agent Traces

Replace `Console` with a trace collector for structured logs:


In [ ]:
async def run_with_trace(team, task: str) -> tuple[TaskResult, list[dict]]:
    """Consume run_stream manually and build a trace list."""
    trace: list[dict] = []
    result = None
    async for event in team.run_stream(task=task):
        if isinstance(event, TaskResult):
            result = event
            trace.append({"event": "TaskResult", "stop_reason": event.stop_reason})
        elif isinstance(event, TextMessage):
            trace.append({
                "event": "TextMessage",
                "source": event.source,
                "preview": event.content[:120],
            })
        else:
            trace.append({"event": type(event).__name__, "source": getattr(event, "source", "?")})
    assert result is not None
    return result, trace


print("run_with_trace ready — call with team.run_stream in live cells")


---

## 7. Logging Configuration


In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")
logger = logging.getLogger("notes_autogen")


def log_task_result(result: TaskResult) -> None:
    summary = summarize_task_result(result)
    logger.info("team_finished stop_reason=%s messages=%d tokens=%s",
                summary["stop_reason"], summary["message_count"], summary["tokens"])
    for i, sp in enumerate(summary["speakers"]):
        logger.debug("  [%d] %s (%s)", i, sp["source"], sp["type"])


log_task_result(sample)


---

## 8. Debugging Speaker Selection Failures

`SelectorGroupChat` uses a **selector agent** or function to pick the next speaker. Failures show up as wrong agents talking or infinite loops.


In [ ]:
def selector_func(messages: Sequence) -> str | None:
    """Explicit selector — easier to debug than model-based selection."""
    if not messages:
        return "docs_writer"
    last = messages[-1]
    if last.source == "docs_writer":
        return "critic"
    if last.source == "critic" and "APPROVE" in getattr(last, "content", ""):
        return None  # let termination handle stop
    return "docs_writer"


selector_team = SelectorGroupChat(
    [docs_writer, critic],
    model_client=model_client,
    selector_func=selector_func,
    termination_condition=MaxMessageTermination(max_messages=8),
)

print("selector_team uses deterministic selector_func — set breakpoints here")


**Common selector bugs:**

1. Selector returns same agent twice → duplicate monologues
2. Selector returns `None` too early → team stalls
3. Model-based selector ignores role descriptions → wire `selector_func` for tests
4. Termination never fires → check `MaxMessageTermination` fallback


---

## 9. Inspecting Selector Decisions


In [ ]:
async def debug_selector_run(team, task: str):
    trace = []
    async for event in team.run_stream(task=task):
        if isinstance(event, TextMessage):
            trace.append(f"SPEAK {event.source}: {event.content[:60]}...")
        elif isinstance(event, TaskResult):
            trace.append(f"STOP {event.stop_reason}")
    return trace


# Dry-run trace format example
print("\\n".join([
    "SPEAK docs_writer: Use Authorization Bearer header [c3]...",
    "SPEAK critic: Correct citation. APPROVE",
    "STOP Text 'APPROVE' mentioned",
]))


---

## 10. `CancellationToken`

Cooperative cancellation for long runs — pass to `run()` / `run_stream()`:


In [ ]:
from autogen_core import CancellationToken


async def run_with_timeout(team, task: str, timeout_sec: float = 30.0):
    token = CancellationToken()

    async def cancel_after():
        await asyncio.sleep(timeout_sec)
        token.cancel()
        print(f"CancellationToken.cancel() after {timeout_sec}s")

    watcher = asyncio.create_task(cancel_after())
    try:
        return await team.run(task=task, cancellation_token=token)
    finally:
        watcher.cancel()


print("Pattern: wire HTTP request disconnect to token.cancel()")


Pair with `ExternalTermination` (**13**) when the UI Stop button should also signal the team.


---

## 11. Compare with LangGraph Observability (**02. LangGraph/15**)

| Question | AutoGen | LangGraph |
|----------|---------|-----------|
| Which node ran? | Message `source` field | `stream_mode="updates"` |
| Final path | `TaskResult.messages` | `state["trace"]` |
| Cancel | `CancellationToken` | `interrupt` / cancel stream |
| Tool visibility | `ToolCallSummaryMessage` | `ToolMessage` |
| SaaS traces | LangSmith (optional) | LangSmith native |


---

## 12. LangSmith Optional Sketch

LangSmith is not AutoGen-native, but you can wrap runs:


In [ ]:
LANGSMITH_SKETCH = '''
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "ls__..."
os.environ["LANGCHAIN_PROJECT"] = "notes-autogen-debug"

from langsmith import traceable

@traceable(name="notes_autogen_team")
async def traced_run(team, question: str) -> dict:
    result = await team.run(task=question)
    return summarize_task_result(result)
'''
print(LANGSMITH_SKETCH)


Wrap at the **service layer** (`run_notes_autogen_service` in **16**) — not inside agent definitions.


---

## 13. Debug Checklist

| Symptom | Check |
|---------|-------|
| Team never stops | `termination_condition` attached? `MaxMessageTermination` fallback? |
| Wrong agent speaks | `selector_func` return value; log each pick |
| Empty final answer | Last `TextMessage` source; tool-only final event? |
| High token usage | `summarize_task_result` tokens; lower `max_messages` |
| Blocked content leaked | Pre-flight `is_blocked()` before `run()` |
| Hung run | `CancellationToken` + `asyncio.wait_for` |


In [ ]:
DEBUG_CHECKLIST = [
    "1. Reproduce with Console(round_robin.run_stream(...))",
    "2. Print summarize_task_result(result)",
    "3. Log each message type and source",
    "4. For Selector — switch to selector_func temporarily",
    "5. Verify termination stop_reason",
    "6. Add CancellationToken timeout",
]
print("\\n".join(DEBUG_CHECKLIST))


---

## 14. API-Ready Debug Response

Shape returned to clients (used in **16**):


In [ ]:
@dataclass
class NotesDebugResponse:
    answer: str
    stop_reason: str
    speakers: list[str]
    tokens: dict[str, int]
    trace_preview: list[str] = field(default_factory=list)


def to_debug_response(result: TaskResult, trace: list[dict]) -> NotesDebugResponse:
    s = summarize_task_result(result)
    return NotesDebugResponse(
        answer=s["last_agent_text"],
        stop_reason=s["stop_reason"],
        speakers=[x["source"] for x in s["speakers"]],
        tokens=s["tokens"],
        trace_preview=[t.get("preview", str(t))[:80] for t in trace[:10]],
    )


demo_resp = to_debug_response(sample, [])
print(demo_resp)


---

## 15. Summary

| Takeaway | Detail |
|----------|--------|
| **`Console` for notebooks** | Fastest way to see agent labels |
| **`TaskResult` for APIs** | `stop_reason` + `messages` + token usage |
| **Log message types** | Especially `ToolCallSummaryMessage` |
| **Debug selectors** | Use explicit `selector_func` |
| **Cancel cooperatively** | `CancellationToken` + timeouts |
| **Next** | **16** — production capstone with `run_notes_autogen_service()` |

Next: **16. Production Patterns for AutoGen**.
